## AI Agent with Memory

market research assistant - that can analyze compnies, industries, and compitators, and then write a report summarizing the findings. The agent should be able to use tools to gather information, and also have memory to remember previous interactions and information it has gathered.

## Setup and Configuration

In [8]:
# Install the specific version of openai-agents
%pip uninstall -y openai-agents
%pip install --upgrade pydantic
%pip install --no-cache-dir openai-agents==0.2.2
%pip install python-dotenv langchain-openai==0.2.1

Found existing installation: openai-agents 0.2.2
Uninstalling openai-agents-0.2.2:
  Successfully uninstalled openai-agents-0.2.2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.1/161.1 kB 5.8 MB/s eta 0:00:00


In [9]:
import os
from openai import OpenAI
from dotenv import load_dotenv
from IPython.display import display, Markdown

# Import required classes from the agents module
# Agent: Represents an AI agent with a specific role and instructions
# Runner: Executes the agent and handles interactions
# SQLiteSession: Provides persistent memory storage for the agent using SQLite
from agents import Agent, Runner, SQLiteSession

In [10]:
load_dotenv()
# Get the OpenAI API key from environment variables or prompt if missing
openai_api_key = os.getenv("OPENAI_API_KEY")
if not openai_api_key:
    from getpass import getpass
    openai_api_key = getpass("OpenAI API key (will not be echoed): ")

# Ensure the agents/OpenAI client can read the key via the environment variable
if openai_api_key:
    os.environ["OPENAI_API_KEY"] = openai_api_key

# Configure the OpenAI Client using our key
openai_client = OpenAI(api_key=openai_api_key)
print("OpenAI client successfully configured.")

OpenAI client successfully configured.


In [11]:
# Define a function for printing markdown cells
def print_markdown(text):
    display(Markdown(text))

In [15]:
# Define the role and instructions for the AI agent
market_researcher_instructions = """
Context:
You are a market research assistant helping analyze companies, industries, and competitors.

Instructions:
When given a question, provide a short factual answer based on your knowledge.

Output:
Start with a verdict prefix: either "✅ FACT:" or "❌ UNKNOWN:"
Follow with a concise one-sentence explanation.
"""

# Create an instance of the Agent
market_researcher_agent = Agent(
    name = "Market Researcher",
    instructions = market_researcher_instructions,
    model = "gpt-5.4-mini"
)

## Agent with no memory

In [13]:
# Example: An AI Agent with NO memory

# Let's give our first question to the AI agent
q1 = "What is the market share of Tesla in the US EV market?"

# Display the user’s question in Markdown format
print_markdown(f"You: '{q1}'")

# Run the agent with the first question (no memory means each query is independent)
resp1 = await Runner.run(starting_agent = market_researcher_agent, input = q1)

# Display the agent’s response
print_markdown(f"🤖 Agent:\n{resp1.final_output}")

You: 'What is the market share of Tesla in the US EV market?'

🤖 Agent:
✅ FACT: Tesla holds around 60-65% of the US electric vehicle market share as of early 2024, making it the dominant EV manufacturer in the country.

In [14]:
# Second question — depends on previous context
# Follow-up question that refers to the previous answer
q2 = "How does that compare to last year?"

# Display the follow-up question
print_markdown(f"\nYou: '{q2}'")

# Run the agent again — since there’s no memory, it does not recall the first question/answer
resp2 = await Runner.run(starting_agent = market_researcher_agent, input = q2)

# Display the agent’s response (will fail to connect it to the first question)
print_markdown(f"🤖 Agent:\n{resp2.final_output}")


You: 'How does that compare to last year?'

🤖 Agent:
❌ UNKNOWN: There is no information or data provided about last year to make a comparison.

## Agent with memory

In [16]:
# Create a session instance
# SQLite-based implementation of session storage.
# This implementation stores conversation history in a SQLite database. 

session = SQLiteSession("conversation")

# First interaction
user_input1 = "What is the market share of Tesla in the US EV market?"
response1 = await Runner.run(
    starting_agent = market_researcher_agent,
    input = user_input1,
    session = session,
)

print_markdown(f"🤖 Agent:\n{response1.final_output}")

🤖 Agent:
✅ FACT: Tesla has historically held the largest share of the U.S. EV market, but its exact market share varies by quarter and year, typically ranging from roughly 45% to 60% in recent years.

In [17]:
# Second interaction with history
user_input2 = "How does that compare to last year?"

response2 = await Runner.run(
    starting_agent = market_researcher_agent,
    input = user_input2,
    session = session,
) 

print_markdown(f"🤖 Agent:\n{response2.final_output}")

🤖 Agent:
✅ FACT: Tesla’s U.S. EV market share has generally declined year over year as more competitors have entered the market, though it still remains the clear leader.